# CN7: 외부 이상탐지 방법을 동일 구간검증에 적용
Mixed gas/shortshot shots count in each task. Three observation blocks, not independently confirmed equipment runs.

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd, torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import confusion_matrix, average_precision_score, precision_recall_curve
torch.set_num_threads(2)
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
t=root/'reports/tables'
d=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[d['product'].eq('CN7')].reset_index(drop=True)
assert len(d)==713 and d.TimeStamp.is_unique
g=d.comparison_block.to_numpy(); blocks=np.unique(g)
cols=['Injection_Time','Filling_Time','Plasticizing_Time','Cycle_Time','Clamp_Close_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Speed','Max_Screw_RPM','Max_Injection_Pressure','Max_Switch_Over_Pressure','Max_Back_Pressure','Average_Back_Pressure']+['Barrel_Temperature_'+str(k) for k in range(1,7)]+['Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4']
a=d[cols].replace([np.inf,-np.inf],np.nan).to_numpy(float)
class VAE(nn.Module):
 def __init__(self,p):
  super().__init__(); self.enc=nn.Sequential(nn.Linear(p,15),nn.ReLU(),nn.Linear(15,5),nn.ReLU());self.mu=nn.Linear(5,2);self.logvar=nn.Linear(5,2);self.dec=nn.Sequential(nn.Linear(2,5),nn.ReLU(),nn.Linear(5,15),nn.ReLU(),nn.Linear(15,p))
 def forward(self,x,sample=True):
  h=self.enc(x);mu=self.mu(h);lv=self.logvar(h).clamp(-10,5);z=mu+torch.randn_like(mu)*torch.exp(.5*lv) if sample else mu
  return self.dec(z),mu,lv
memo={}; audits=[]
def fit_score(excluded):
 key=tuple(sorted(excluded))
 if key in memo:return memo[key]
 normal=np.flatnonzero(d.PassOrFail.eq(0).to_numpy() & ~np.isin(g,key))
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(a[normal]);x=imp.transform(a)
 keep=x[normal].std(axis=0)>1e-7
 sc=StandardScaler().fit(x[normal][:,keep]);x=np.clip(sc.transform(x[:,keep]),-20,20)
 models=[LedoitWolf().fit(x[normal]),IsolationForest(n_estimators=150,random_state=42,n_jobs=2).fit(x[normal]),OneClassSVM(nu=.01,gamma='scale').fit(x[normal]),LocalOutlierFactor(n_neighbors=20,novelty=True).fit(x[normal])]
 raw=[models[0].mahalanobis(x),-models[1].score_samples(x),-models[2].score_samples(x),-models[3].score_samples(x)]
 torch.manual_seed(42);m=VAE(x.shape[1]);opt=torch.optim.Adam(m.parameters(),lr=.001,weight_decay=.0001);xt=torch.tensor(x[normal],dtype=torch.float32)
 m.train()
 for epoch in range(80):
  for idx in torch.randperm(len(xt)).split(64):
   xb=xt[idx];re,mu,lv=m(xb);loss=((re-xb)**2).mean()+.01*(-.5*(1+lv-mu.square()-lv.exp()).sum(1)).mean()
   opt.zero_grad();loss.backward();opt.step()
 m.eval()
 with torch.no_grad():
  allx=torch.tensor(x,dtype=torch.float32);re,_,_=m(allx,sample=False);raw.append(((re-allx)**2).mean(1).numpy())
 # All rank references come only from reference normal rows, never heldout rows.
 rank=np.column_stack([np.searchsorted(np.sort(s[normal]),s,side='right')/(len(normal)+1) for s in raw])
 out={'mahalanobis':rank[:,0],'vae':rank[:,4],'diverse_mean':rank[:,[1,2,3,4]].mean(1)}
 votes=(rank[:,[1,2,3,4]]>=.99).sum(1)
 for k in range(1,5):out['consensus_'+str(k)]=(votes>=k).astype(float)
 out=pd.DataFrame(out)
 audits.append(dict(excluded_blocks='|'.join(key),normal_train=len(normal),feature_count=int(keep.sum()),overlap_rows=int(np.isin(g[normal],key).sum()),used_defect_train=0))
 memo[key]=out
 return out
def cut_f1(y,s):
 if y.sum()==0:return float(np.quantile(s,.99)),0.0
 p,r,c=precision_recall_curve(y,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12);k=max(range(len(c)),key=lambda k:(f[k],r[k],c[k]));return float(c[k]),float(f[k])
records=[];choices=[]
for task,reason,expected in [('gas','가스',13),('shortshot','미성형',3),('startup','초기허용불량',5),('all_defects','.',19)]:
 mask=d.PassOrFail.eq(0)|d.Reason.str.contains(reason,na=False); ids=np.flatnonzero(mask)
 y=(d.PassOrFail.eq(1) & d.Reason.str.contains(reason,na=False)).to_numpy(int);assert y[ids].sum()==expected
 for block in blocks:
  q=ids[g[ids]==block]; ref=ids[g[ids]!=block];outer=fit_score([block]).iloc[q]
  inner=pd.DataFrame(index=ref,columns=outer.columns,dtype=float)
  for ib in np.unique(g[ref]):
   qi=ref[g[ref]==ib];inner.loc[qi]=fit_score([block,ib]).iloc[qi].to_numpy()
  candidates=[]
  for name in outer.columns:
   if name.startswith('consensus'):cut=.5;pred=inner[name].to_numpy()>=cut;tp=((y[ref]==1)&pred).sum();fp=((y[ref]==0)&pred).sum();fn=((y[ref]==1)&~pred).sum();fi=2*tp/max(2*tp+fp+fn,1)
   else:cut,fi=cut_f1(y[ref],inner[name].to_numpy())
   candidates.append((fi,name,cut))
   for i,s in zip(q,outer[name]):records.append(dict(task=task,model=name,TimeStamp=d.TimeStamp.iloc[i],block=block,actual=int(y[i]),score=float(s),threshold=cut,prediction=int(s>=cut)))
  fi,name,cut=max(candidates,key=lambda z:(z[0],z[1])) if y[ref].sum()>0 else next(z for z in candidates if z[1]=='diverse_mean')
  choices.append(dict(task=task,block=block,model=name,inner_F1=fi,threshold=cut))
  for i,s in zip(q,outer[name]):records.append(dict(task=task,model='primary_inner_selected',TimeStamp=d.TimeStamp.iloc[i],block=block,actual=int(y[i]),score=float(s),threshold=cut,prediction=int(s>=cut)))
  print(task,block,'selected',name,flush=True)
p=pd.DataFrame(records);rows=[]
for (task,name),z in p.groupby(['task','model']):
 assert z.TimeStamp.is_unique and len(z)=={'gas':707,'shortshot':697,'startup':699,'all_defects':713}[task]
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(task=task,model=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/max(2*tp+fp+fn,1),AP=average_precision_score(z.actual,z.score)))
metrics=pd.DataFrame(rows)
p.to_csv(t/'cn7_external_methods_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(t/'cn7_external_methods_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(t/'cn7_external_methods_inner_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audits).to_csv(t/'cn7_external_methods_training_audit.csv',index=False,encoding='utf-8-sig')
p[p.actual.eq(1)].to_csv(t/'cn7_external_methods_all_defects.csv',index=False,encoding='utf-8-sig')
print(metrics.to_string(index=False),flush=True)
report='# CN7 외부 방법 적용 — 정상 학습 및 중첩 구간검증\n\n'
report+='기존 CN7 713 Shot에서 정상694+가스13, 정상694+미성형3, 정상694+초기5, 정상694+전체불량19를 각각 평가. 가스·미성형 혼합 Shot2개는 가스/미성형 각각의 평가에서 양성으로 포함. 다른 불량은 해당 평가에서 제외. RH/LH 중복 Shot을 다시 늘리지 않았다.\n\n'
report+='3개 observation block 외부검증과 2개 내부구간검증. 연속 설비 run 확정값이 아닌 기존 관측 구간이다. 모든 정규화·공분산·신경망·점수 기준은 학습 정상만 사용. 임계값/모델 선택은 내부 OOF만 사용. 외부 불량 학습 및 Timestamp 중복 0. 이미 반복 열람한 자료라 새 데이터 독립검증은 아니다.\n\n'
report+='22개 원본 센서 후보에서 학습 정상 상수만 제거. Average_Screw_RPM은 기존 단위 이슈로 제외. 센서 입력은 위 노트북 cols에 명시.\n\n'
report+='Mahalanobis는 원본 역행렬 대신 LedoitWolf 축소 공분산. VAE는 22→15→5→mu/logvar 2→5→15→22, MSE+0.01 KL,80epoch,정상만 학습. 평가는 eval 및 잠재 평균으로 결정론적 복원. IF150트리,OCSVM nu.01,LOF20이웃. 합의는 4개(IF/SVM/LOF/VAE) 학습 정상99% 기준 통과 수 1~4; 원본 AE 합의를 수정 VAE로 대체한 비교이며 원본 성적의 재현은 아니다.\n\n'
report+='출처: https://github.com/johnwslee/injection_molding_analysis ; https://github.com/tkddnjs-dlqslek/smartfactory-xai . 원본 코드의 평가 라벨 임계값 선정, KL 누락 등을 그대로 복사하지 않았다. 비라벨 데이터는 이번에는 사용하지 않았다.\n\n'
report+=metrics.to_csv(index=False)
report+='\nprimary_inner_selected만 사전에 정의한 내부 선택 파이프라인 결과. 개별 후보 중 외부 성적 최고를 재선택하면 탐색 결과다. 불량 유형이 특정 구간에 몰릴 수 있고 일부 내부 학습/평가 구간에는 해당 불량이 없어 임계값 선택이 불안정할 수 있다. 전체불량은 초기불량까지 섞여 있어 가스 성적으로 해석하면 안 된다.\n'
(root/'reports/cn7_external_methods_comparison.md').write_text(report,encoding='utf-8')


gas 20201016_045747_CN7 selected diverse_mean


gas 20201019_232446_CN7 selected consensus_4


gas 20201027_001040_CN7 selected diverse_mean


shortshot 20201016_045747_CN7 selected diverse_mean


shortshot 20201019_232446_CN7 selected consensus_4


...

all_defects 20201016_045747_CN7 selected mahalanobis


all_defects 20201019_232446_CN7 selected consensus_4


all_defects 20201027_001040_CN7 selected diverse_mean


       task                  model  TP  FP  FN  TN       F1       AP
all_defects            consensus_1  19 694   0   0 0.051913 0.026648
all_defects            consensus_2  19 497   0 197 0.071028 0.036822
all_defects            consensus_3  19 287   0 407 0.116923 0.062092
all_defects            consensus_4  13 134   6 560 0.156627 0.068924
all_defects           diverse_mean   8 262  11 432 0.055363 0.421841
all_defects            mahalanobis  19 592   0 102 0.060317 0.335697
all_defects primary_inner_selected  19 198   0 496 0.161017 0.078138
all_defects                    vae  14 122   5 572 0.180645 0.120089
        gas            consensus_1  13 694   0   0 0.036111 0.018388
        gas            consensus_2  13 497   0 197 0.049713 0.025490
        gas            consensus_3  13 287   0 407 0.083067 0.043333
        gas            consensus_4   7 134   6 560 0.090909 0.035219
        gas           diverse_mean   4 264   9 430 0.028470 0.186388
        gas            mahalanobis

3408